# Notebook 06: Comprehensive Model Comparison (All 3 Principal Components)

This notebook evaluates the same LOSO benchmark across two target families: a toy mean-ratings target for quick experimentation, and the proper Daly PCA targets representing Valence, Energy, and Tension. For each target family, the models are compared on three feature subsets: both EEG + audio, EEG only, and audio only.

In [ ]:
import os
import glob
import logging
import pandas as pd
import numpy as np
from IPython.display import display
from sklearn.base import clone
from sklearn.impute import KNNImputer
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Lasso
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
from xgboost import XGBRegressor
from tqdm.auto import tqdm
import warnings

warnings.filterwarnings('ignore')

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s %(levelname)s %(message)s',
)
logger = logging.getLogger('emotion_benchmark')

In [ ]:
# Load features
base_path = os.path.abspath(os.path.join(os.getcwd(), '..')) if os.path.split(os.getcwd())[-1] == 'notebooks' else os.getcwd()
features_dir = os.path.join(base_path, 'data', 'features')
csv_files = glob.glob(os.path.join(features_dir, 'sub-*_features.csv'))
full_df = pd.concat([pd.read_csv(f) for f in csv_files], ignore_index=True)
print(f"Total trials collected: {len(full_df)}")

# Build the two target families
rating_cols = [f'Q{q}' for q in range(800, 808)]
imputer = KNNImputer(n_neighbors=5, weights='distance')
imputed_ratings = imputer.fit_transform(full_df[rating_cols])

full_df['Mean_Rating_Toy'] = imputed_ratings.mean(axis=1)

pca = PCA(n_components=3)
target_b_pca = pca.fit_transform(imputed_ratings)
full_df['PC1_Valence_Subj'] = target_b_pca[:, 0]
full_df['PC2_Energy_Subj'] = target_b_pca[:, 1]
full_df['PC3_Tension_Subj'] = target_b_pca[:, 2]
print(f"Explained Variance by 3 PCs: {sum(pca.explained_variance_ratio_):.2f}")

# Separate EEG and audio feature groups
meta_cols = {
    'track_id', 'run', 'subject', 'number', 'valence', 'energy', 'tension',
    'anger', 'fear', 'happy', 'sad', 'tender', 'TARGET',
    'Mean_Rating_Toy', 'PC1_Valence_Subj', 'PC2_Energy_Subj', 'PC3_Tension_Subj',
} | set(rating_cols)

eeg_prefixes = (
    'FP1_', 'FP2_', 'F7_', 'F3_', 'Fz_', 'F4_', 'F8_',
    'T3_', 'C3_', 'Cz_', 'C4_', 'T4_', 'T5_', 'P3_', 'Pz_', 'P4_', 'T6_', 'O1_', 'O2_'
)
audio_prefixes = ('mfcc_', 'chroma_', 'spectral_', 'rms_energy')

feature_columns = [c for c in full_df.columns if c not in meta_cols]
eeg_columns = [c for c in feature_columns if c.startswith(eeg_prefixes)]
audio_columns = [c for c in feature_columns if c.startswith(audio_prefixes)]
both_columns = sorted(set(eeg_columns) | set(audio_columns))

feature_sets = {
    'Both': both_columns,
    'EEG only': eeg_columns,
    'Audio only': audio_columns,
}

print(f"Feature counts -> Both: {len(both_columns)}, EEG only: {len(eeg_columns)}, Audio only: {len(audio_columns)}")

subjects = full_df['subject'].unique()
models = {
    'Lasso': Lasso(alpha=0.1, random_state=42, max_iter=10000),
    'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1),
    'XGBoost': XGBRegressor(
        n_estimators=100,
        learning_rate=0.1,
        random_state=42,
        n_jobs=-1,
        tree_method='hist',
    ),
}

def evaluate_loso(estimator, X_frame, y_series, model_name, target_name, feature_name):
    actuals = []
    predictions = []

    fold_bar = tqdm(subjects, desc=f"{target_name} | {model_name} | {feature_name}", leave=False)
    for test_sub in fold_bar:
        train_idx = full_df['subject'] != test_sub
        test_idx = full_df['subject'] == test_sub

        X_train = X_frame.loc[train_idx]
        X_test = X_frame.loc[test_idx]
        y_train = y_series.loc[train_idx]
        y_test = y_series.loc[test_idx]

        fitted_model = clone(estimator)

        if model_name == 'Lasso':
            scaler = StandardScaler()
            X_train_fit = scaler.fit_transform(X_train)
            X_test_fit = scaler.transform(X_test)
        else:
            X_train_fit = X_train.values
            X_test_fit = X_test.values

        fitted_model.fit(X_train_fit, y_train)
        fold_predictions = fitted_model.predict(X_test_fit)

        actuals.extend(y_test.to_numpy())
        predictions.extend(np.asarray(fold_predictions))

    actuals = np.asarray(actuals)
    predictions = np.asarray(predictions)

    rmse = float(np.sqrt(mean_squared_error(actuals, predictions)))
    r2 = float(r2_score(actuals, predictions))
    pearson_r = float(np.corrcoef(actuals, predictions)[0, 1])

    return {
        'RMSE': rmse,
        'R2': r2,
        'Pearson r': pearson_r,
    }

def build_results_table(target_map, family_name):
    rows = []

    logger.info('Starting benchmark family: %s', family_name)
    for target_name, y_series in target_map.items():
        logger.info('Target: %s', target_name)
        for model_name, estimator in models.items():
            logger.info('Model: %s', model_name)
            row = {
                'Family': family_name,
                'Target': target_name,
                'Model': model_name,
            }

            for feature_name, columns in feature_sets.items():
                logger.info('Evaluating %s / %s / %s', target_name, model_name, feature_name)
                metrics = evaluate_loso(
                    estimator=estimator,
                    X_frame=full_df[columns],
                    y_series=y_series,
                    model_name=model_name,
                    target_name=target_name,
                    feature_name=feature_name,
                )
                row[f'{feature_name} RMSE'] = metrics['RMSE']
                row[f'{feature_name} R2'] = metrics['R2']
                row[f'{feature_name} r'] = metrics['Pearson r']

            rows.append(row)

    results = pd.DataFrame(rows)
    summary = results[[
        'Family', 'Target', 'Model',
        'Both r', 'EEG only r', 'Audio only r',
    ]].copy()
    return results, summary

mean_target_map = {
    'Mean_Rating_Toy': full_df['Mean_Rating_Toy'],
}

daly_target_map = {
    'PC1_Valence_Subj': full_df['PC1_Valence_Subj'],
    'PC2_Energy_Subj': full_df['PC2_Energy_Subj'],
    'PC3_Tension_Subj': full_df['PC3_Tension_Subj'],
}

mean_results, mean_summary = build_results_table(mean_target_map, 'Mean ratings experiment')
daly_results, daly_summary = build_results_table(daly_target_map, 'Daly PCA targets')

print('\n=== Mean ratings experiment: Pearson summary ===')
display(mean_summary)
print('\n=== Mean ratings experiment: full metrics ===')
display(mean_results)

print('\n=== Daly PCA targets: Pearson summary ===')
display(daly_summary)
print('\n=== Daly PCA targets: full metrics ===')
display(daly_results)

Feature matrix shape (X): (990, 199)


In [3]:
models = {
    'Lasso (Linear)': Lasso(alpha=0.1, random_state=42, max_iter=10000),
    'Random Forest (Bagged)': RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1),
    'XGBoost (Boosting)': XGBRegressor(n_estimators=100, learning_rate=0.1, random_state=42, n_jobs=-1)
}
targets = ['PC1_Valence_Subj', 'PC2_Energy_Subj', 'PC3_Tension_Subj']
results_table = []

for pc in targets:
    row = {'Response PCs': pc}
    for model_name, model in models.items():
        actuals = []
        predictions = []
        for test_sub in subjects:
            train_idx = full_df['subject'] != test_sub
            test_idx = full_df['subject'] == test_sub
            X_train, y_train = X[train_idx], full_df.loc[train_idx, pc]
            X_test, y_test = X[test_idx], full_df.loc[test_idx, pc]
            
            if 'Lasso' in model_name:
                scaler = StandardScaler()
                X_train_final = scaler.fit_transform(X_train)
                X_test_final = scaler.transform(X_test)
            else:
                X_train_final, X_test_final = X_train.values, X_test.values
            
            model.fit(X_train_final, y_train)
            preds = model.predict(X_test_final)
            predictions.extend(preds)
            actuals.extend(y_test)
        
        corr = np.corrcoef(actuals, predictions)[0,1]
        row[model_name] = f"{corr:.4f}"
    results_table.append(row)

df_results = pd.DataFrame(results_table)
display(df_results)

KeyboardInterrupt: 